# Similarity metrics

In [4]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from datetime import datetime, timedelta

import random

import warnings
warnings.filterwarnings("ignore")

import similaritymeasures

plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Data loading

In [2]:
df = pd.read_csv("hsa_state_new.csv", parse_dates=["time_value"])
df = df.dropna(subset=["state", "hsa_value", "state_value"])
df = df.sort_values(["hsa_id", "time_value"]).reset_index(drop=True)

print(f"Rows       : {len(df):,}")
print(f"States     : {df['state'].nunique()}")
print(f"HSAs       : {df['hsa_id'].nunique()}")
print(f"Date range : {df['time_value'].min().date()} to {df['time_value'].max().date()}")
df.head()

Rows       : 125,839
States     : 38
HSAs       : 671
Date range : 2022-09-25 to 2026-05-17


,time_value,hsa_id,state,hsa_value,state_value,hsa_pop,state_pop,pop_ratio
0,2022-09-25,1,Maryland,0.00,0.26,96475.0,6170738.0,0.015634
1,2022-10-02,1,Maryland,0.00,0.41,96475.0,6170738.0,0.015634
2,2022-10-09,1,Maryland,0.00,0.93,96475.0,6170738.0,0.015634
3,2022-10-16,1,Maryland,0.19,1.80,96475.0,6170738.0,0.015634
4,2022-10-23,1,Maryland,0.70,3.74,96475.0,6170738.0,0.015634


In [7]:
hsa_list = df["hsa_id"].values
chosen_id = random.choice(hsa_list)

hsa_display = df[df['hsa_id'] == chosen_id]

dates = hsa_display["time_value"].values
hsa_curve = hsa_display["hsa_value"].values
state_curve = hsa_display["state_value"].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=hsa_curve, name='HSA'))
fig.add_trace(go.Scatter(x=dates, y=state_curve, name='State'))
fig.update_layout(title=f"HSA {chosen_id} ({hsa_display['state'].iloc[0]})")
fig.show()

## Helper functions

In [ ]:
def make_curve(t_days, values):
    curve = np.zeros((len(t_days), 2))
    curve[:, 0] = t_days
    curve[:, 1] = values
    return curve

print("Helpers ready")

Helpers ready


## HSA analysis
### Window specification

In [ ]:
records = []

groups = list(df.groupby("hsa_id"))
n_total = len(groups)

for idx, (hsa_id, grp) in enumerate(groups):

    if idx % 100 == 0:
        print(f"  {idx}/{n_total} HSAs processed...")

    grp = grp.sort_values("time_value")
    if len(grp) < 4:
        continue

    t_days     = (grp["time_value"] - grp["time_value"].min()).dt.days.values.astype(float)
    state_vals = grp["state_value"].values.astype(float)
    hsa_vals   = grp["hsa_value"].values.astype(float)

    # exp_data = ground truth (state), num_data = local signal (hsa)
    exp_data = make_curve(t_days, state_vals)
    num_data = make_curve(t_days, hsa_vals)

    area      = similaritymeasures.area_between_two_curves(exp_data, num_data)
    rmse       = math.sqrt(similaritymeasures.mse(exp_data, num_data))

    path = warp_path_from_matrix(dtw_matrix)
    lags = lag_metrics(path)

    records.append({
        "hsa_id"             : hsa_id,
        "state"              : grp["state"].iloc[0],
        "n_obs"              : len(grp),
        "pop_ratio"          : grp["pop_ratio"].iloc[0],
        "abc"                : area,
        "dtw"                : dtw_dist,
        "rmse"                : rmse,
        "timing_component"   : timing_component,
        "magnitude_component": dtw_dist,
        "timing_share"       : timing_share,
        **lags,
    })

hsa_df = pd.DataFrame(records)
print(f"Done — {len(hsa_df)} HSAs")
hsa_df.round(4).head(10)

  0/671 HSAs processed...
  100/671 HSAs processed...
  200/671 HSAs processed...
  300/671 HSAs processed...
  400/671 HSAs processed...
  500/671 HSAs processed...
  600/671 HSAs processed...
Done — 671 HSAs


,hsa_id,state,n_obs,pop_ratio,abc,dtw,rmse,timing_component,magnitude_component,timing_share,mean_lag,mean_abs_lag,lag_variance,max_abs_lag
0,1,Maryland,189,0.0156,908.145,129.91,0.9390,778.235,129.91,0.8570,0.0,0.0,0.0,0.0
1,2,Kentucky,189,0.0996,585.550,83.77,0.6093,501.780,83.77,0.8569,0.0,0.0,0.0,0.0
2,3,Delaware,189,0.4303,312.865,44.75,0.3091,268.115,44.75,0.8570,0.0,0.0,0.0,0.0
3,5,Virginia,189,0.0634,804.965,115.25,0.7654,689.715,115.25,0.8568,0.0,0.0,0.0,0.0
4,7,West Virginia,189,0.1667,543.620,77.84,0.5961,465.780,77.84,0.8568,0.0,0.0,0.0,0.0
5,8,Pennsylvania,105,0.0258,228.410,32.72,0.4581,195.690,32.72,0.8567,0.0,0.0,0.0,0.0
6,9,Maine,189,0.3047,412.825,59.03,0.4658,353.795,59.03,0.8570,0.0,0.0,0.0,0.0
7,10,New York,189,0.0239,448.910,64.22,0.4285,384.690,64.22,0.8569,0.0,0.0,0.0,0.0
8,11,Kentucky,189,0.0154,969.500,138.59,0.9438,830.910,138.59,0.8571,0.0,0.0,0.0,0.0
9,12,Illinois,189,0.0022,1969.870,281.54,2.3384,1688.330,281.54,0.8571,0.0,0.0,0.0,0.0


In [ ]:
hsa_df.to_csv("hsa_metrics.csv", index=False)
print("Saved: hsa_metrics.csv")

# SoftDTW
# SimilarityMetrics
# Analysis by HSA
# Analysis by State